# 📝 과제 01: 의약품 상담 도우미에 GraphRAG 적용하기

## 만들 결과

약국과 제약사 상담 창구에서 쓰는 **의약품 상담 도우미**의 검색 부분을 Microsoft GraphRAG로 만듭니다. 식약처 「e약은요」 안내문 18편을 색인하고, 손님 질문 8개를 질문 유형에 맞는 검색으로 실행해 정답 근거 문서를 넘겼는지와 비용을 비교합니다.

- 의약품에 맞는 개체 유형과 한국어 색인 프롬프트로 설정을 만들고 색인합니다.
- 질문 유형마다 고른 검색을 실행하고, 검색이 넘긴 문서 ID로 결과 표 `output/gwaje01_results.csv`를 저장합니다.
- 새 안내문 한 편을 추가해 색인을 갱신하고, 처음 색인과 비용을 비교합니다.

교안 01의 색인과 네 가지 검색을 의약품 자료에 적용합니다. 의학적 조언이 아니라 수업용 검색 자료이며, 실제 복용은 의사·약사와 상의해야 합니다.

<img src="images/gwaje01_overview.png" width="1050" alt="의약품 상담 도우미의 GraphRAG 흐름: 안내문 18편에서 LLM이 제품·성분·제조사·증상·주의 대상 개체와 관계를 뽑아 그래프와 커뮤니티 보고서를 만들고, 손님 질문을 유형별 검색으로 보냅니다. 단일 사실은 Basic·Local로 안내문 한 편을, 멀티홉은 제조사·성분 개체를 다리로 두 안내문을 잇는 DRIFT로, 전역은 커뮤니티 보고서 전체를 요약하는 Global로 찾고, 넘긴 문서와 호출 수를 결과 표에 기록합니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/과제_01/`의 의약품 안내문 19편과 골드 질문 8개(단일 사실 3, 멀티홉 3, 전역 2). 비교는 18편으로 하고, `extra.json`이 가리키는 1편은 갱신 때 추가합니다. 출처는 [자료 설명](data/과제_01/README.md)에 있습니다.
- **순서:** 개체 유형 -> 한국어 프롬프트 -> 설정 dict -> 문서 ID 변환 -> 검색 계획 -> 색인 -> 골드 질문 검색 -> 갱신 -> 서술
- **제공:** 환경 설정, 자료 읽기, 프로젝트 폴더와 원문 비율, 기본 프롬프트와 `korean_rule`, 모델·경로 기본 설정, 사용량 콜백, `read_tables`, `run_search`, `make_result_row`
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 자가채점 셀을 실행하세요. 색인·검색·갱신 셀만 모델을 호출하며 몇 분씩 걸립니다.

In [ ]:
# 환경 설정: 기본 라이브러리, 경로, API 키. 나머지 라이브러리는 처음 쓰는 셀에서 불러옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# material_dir는 data·output이 있는 단원 폴더입니다. 정답에서는 ".."을 사용합니다.
material_dir = Path(".").resolve()
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 노트북 폴더부터 위로 올라가며 처음 찾은 .env를 읽고, 없는 키는 저장소 최상위 .env에서 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.parent / ".env")


def read_json(name):
    """data 폴더 아래 name 경로의 JSON 파일을 읽어 list 또는 dict로 반환합니다.

    name: data 폴더 기준 상대 경로(예: "과제_01/questions.json")
    """
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# 답변 모델(.env에 없으면 gpt-5.6-luna)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
# 답변 모델에 넘길 문서 수
TOP_K = 5

print("답변 모델:", model_name)

In [ ]:
# 갱신 때 추가할 1편은 비교 세트에서 뺍니다.
extra_id = read_json("과제_01/extra.json")["doc_id"]
# 한 줄에 문서 하나인 JSON Lines 파일이라 lines=True로 읽습니다.
docs_table = pd.read_json(data_dir / "과제_01" / "docs.jsonl", lines=True)
docs_table = docs_table[docs_table["doc_id"] != extra_id]

print("비교 세트 문서:", len(docs_table), "편 | 분류별 문서 수:", docs_table["category"].value_counts().to_dict())
display(docs_table[["doc_id", "title", "manufacturer", "category"]])

In [ ]:
# 골드 질문 8개(id, qtype, question, gold_doc_ids)
questions = read_json("과제_01/questions.json")
# 질문 id로 찾도록 dict로 만듭니다.
questions_by_id = {question["id"]: question for question in questions}
question_table = pd.DataFrame(questions)

print("유형별 질문 수:", question_table["qtype"].value_counts().to_dict())
display(question_table[["id", "qtype", "question", "gold_doc_ids"]])

GraphRAG 프로젝트 폴더 `output/gwaje01_graphrag/`를 새로 만들고 문서 18편을 `input/`에 넣습니다. 이전 색인과 캐시도 지우므로, 색인을 처음부터 다시 하려면 이 셀부터 실행합니다. Local·DRIFT가 원문에 쓸 비율도 여기서 계산합니다.

In [ ]:
import shutil

import tiktoken

project_dir = output_dir / "gwaje01_graphrag"
input_dir = project_dir / "input"
# 이전 색인·캐시·입력을 폴더째 지웁니다. ignore_errors=True면 폴더가 없어도 오류가 나지 않습니다.
shutil.rmtree(project_dir, ignore_errors=True)
input_dir.mkdir(parents=True)

# 문서 18편을 input에 복사합니다. 파일 이름이 문서 제목이 되므로 .txt를 떼면 문서 ID입니다.
for doc_id in docs_table["doc_id"]:
    shutil.copy(data_dir / "과제_01" / "docs" / f"{doc_id}.txt", input_dir)

# GraphRAG와 같은 토크나이저로 문서 토큰 수를 셉니다.
encoding = tiktoken.get_encoding("o200k_base")
input_files = sorted(input_dir.glob("*.txt"))
total_tokens = 0

for path in input_files:
    total_tokens += len(encoding.encode(path.read_text(encoding="utf-8")))

# 원문 비율 = 문서 TOP_K편 분량 토큰(원문 테이블 머리글 몫 50 포함) / Local 컨텍스트 한도(기본값 12,000)
text_unit_prop = round((TOP_K * total_tokens / len(input_files) + 50) / 12000, 3)

print("입력 문서:", len(input_files), "편 | 전체 토큰:", total_tokens, "| 원문 비율(text_unit_prop):", text_unit_prop)

## 1. 의약품 안내문에 맞는 개체 유형을 정하세요

GraphRAG는 `entity_types`에 적은 유형으로 개체를 뽑습니다. 멀티홉 질문은 제품에서 제조사나 성분을 거쳐 다른 제품으로 건너가므로, 이 다리가 개체로 뽑혀야 그래프로 이어집니다.

**작성:** 개체 유형 목록 `entity_types`

**목적:** 교안 01에서 관광 안내문에 장소·인물·단체·행사를 정했듯이, 의약품 안내문에서 뽑을 개체 유형을 정합니다.

**구현할 처리**
1. 안내문에 자주 나오는 대상을 유형 이름 4~6개로 적습니다(예: 효능이 가리키는 대상, 복용하면 안 되는 사람). 이름은 한글로만 씁니다. 띄어쓰기는 되지만 영문·괄호는 넣지 않습니다.
2. 멀티홉의 다리인 `제조사`와 `성분`은 이 이름 그대로 넣습니다.

**반환값:** `entity_types`(문자열 리스트). 교안의 관광 안내문은 `["장소", "인물", "단체", "행사"]`였습니다.

In [ ]:
# [작성] 의약품 안내문에서 뽑을 개체 유형을 4~6개 정하세요.
entity_types = ...

In [ ]:
# [자가채점] 유형 개수·표기와 멀티홉 다리 유형을 확인합니다.
import re

assert isinstance(entity_types, list), "entity_types에 문자열 리스트를 담으세요. `...`를 채웠는지도 확인하세요."
assert 4 <= len(entity_types) <= 6, "유형은 4~6개로 정하세요. 너무 많으면 같은 대상이 여러 유형으로 흩어집니다."
assert len(set(entity_types)) == len(entity_types), "같은 유형을 두 번 적지 마세요."

for entity_type in entity_types:
    assert isinstance(entity_type, str) and re.fullmatch(r"[가-힣 ]+", entity_type), f"{entity_type!r}: 유형 이름은 한국어로 적으세요."

assert "제조사" in entity_types, "멀티홉 질문은 제조사로 이어집니다. `제조사`를 유형 이름으로 넣으세요."
assert "성분" in entity_types, "멀티홉 질문은 성분으로도 이어집니다. `성분`을 유형 이름으로 넣으세요."

print("개체 유형 확인 완료:", entity_types)

## 2. 색인 프롬프트 세 개를 한국어로 바꾸세요

교안 01에서 본 대로 GraphRAG에는 출력 언어 설정이 없습니다. 기본 추출 프롬프트 `GRAPH_EXTRACTION_PROMPT`는 영어로 출력하라고 지시해서, 그대로 쓰면 「판콜에스내복액」 같은 이름이 영어로 바뀌어 질문의 한국어 이름과 이어지지 않습니다. 설명 요약(`SUMMARIZE_PROMPT`)과 커뮤니티 보고서(`COMMUNITY_REPORT_PROMPT`) 프롬프트는 언어 지시가 없어 맨 앞에 한국어 지시 `korean_rule`을 붙입니다.

**작성:** 바꾼 추출 프롬프트 `extract_prompt`, 세 프롬프트 dict `prompt_texts`, 저장 경로 dict `prompt_paths`

**목적:** 개체 이름은 원문 표기 그대로, 설명과 커뮤니티 보고서는 한국어로 나오는 프롬프트를 파일로 저장합니다. 설정이 프롬프트를 파일 경로로만 받기 때문입니다.

**구현할 처리**
1. `GRAPH_EXTRACTION_PROMPT`에서 `"Return output in English as"`만 `replace`로 바꿔 `extract_prompt`에 담습니다. 바꿀 문장에는 설명을 한국어로 쓰라는 지시(Write all descriptions in Korean)와 개체 이름을 번역하지 말고 원문 표기 그대로 두라는 지시를 넣고, `Return output as`로 끝맺습니다.
2. `prompt_texts`의 `summarize_descriptions`·`community_report` 값은 기본 프롬프트 맨 앞에 `korean_rule`을 붙인 문자열입니다. 기본 프롬프트 내용은 고치지 않습니다.
3. `prompt_texts`의 이름마다 `output_dir / f"gwaje01_{name}_ko.txt"`에 UTF-8로 저장하고, 그 경로를 `prompt_paths[name]`에 담습니다.

**반환값:** `prompt_paths`(`extract_graph`, `summarize_descriptions`, `community_report` 세 키와 파일 경로의 dict). 예: `prompt_paths["extract_graph"]`는 `output_dir / "gwaje01_extract_graph_ko.txt"`입니다. 프롬프트 안의 `{entity_types}`·`{input_text}` 같은 중괄호 자리는 GraphRAG가 값을 채우는 곳이라 그대로 둡니다.

In [ ]:
# graphrag 패키지에 든 기본 프롬프트 문자열 세 개를 불러옵니다.
from graphrag.prompts.index.community_report import COMMUNITY_REPORT_PROMPT
from graphrag.prompts.index.extract_graph import GRAPH_EXTRACTION_PROMPT
from graphrag.prompts.index.summarize_descriptions import SUMMARIZE_PROMPT

# 추출 프롬프트에서 바꿀 문장이 든 줄을 봅니다.
for line in GRAPH_EXTRACTION_PROMPT.splitlines():
    if "Return output in English as" in line:
        print(line)

# 설명 요약·커뮤니티 보고서 프롬프트 맨 앞에 붙일 한국어 지시
korean_rule = "Write all text values in Korean.\n\n"

In [ ]:
# [작성] 색인 프롬프트 세 개를 한국어로 바꿔 파일로 저장하세요.
# 1. 추출 프롬프트: "Return output in English as"를 한국어 지시로 바꾸세요.
extract_prompt = ...

# 2. 설명 요약·커뮤니티 보고서 프롬프트: 기본 프롬프트 맨 앞에 korean_rule을 붙이세요.
prompt_texts = {"extract_graph": extract_prompt,
                "summarize_descriptions": ...,
                "community_report": ...}

# 3. 이름마다 output_dir / f"gwaje01_{name}_ko.txt"에 저장하고 경로를 prompt_paths에 담으세요.
prompt_paths = {}
...

In [ ]:
# [자가채점] 언어 지시만 바뀌고 나머지 프롬프트와 값 자리가 그대로인지, 파일로 저장했는지 확인합니다.
assert isinstance(extract_prompt, str), "extract_prompt에 바꾼 추출 프롬프트 문자열을 담으세요. `...`를 채웠는지도 확인하세요."
assert "Return output in English" not in extract_prompt, "추출 프롬프트의 영어 출력 문장을 바꾸세요."
head, tail = GRAPH_EXTRACTION_PROMPT.split("Return output in English as")
assert extract_prompt.startswith(head) and extract_prompt.endswith(tail), "바꿀 문장 앞뒤는 그대로 두세요. replace로 그 문장만 바꿉니다."
instruction = extract_prompt[len(head):len(extract_prompt) - len(tail)]
assert "Korean" in instruction or "한국어" in instruction, "바꾼 문장에 한국어로 쓰라는 지시를 넣으세요."
assert "Return output" in instruction, "뒤에 이어지는 목록 형식 지시가 끊기지 않게 바꾼 문장을 Return output as로 끝맺으세요."
assert "translat" in instruction.lower() or "as written" in instruction.lower() or "번역" in instruction, "개체 이름은 번역하지 말고 원문 표기 그대로 두라는 지시를 넣으세요."

# 기본 프롬프트와 이름을 짝지어 나머지 두 프롬프트를 봅니다.
defaults = {"summarize_descriptions": SUMMARIZE_PROMPT, "community_report": COMMUNITY_REPORT_PROMPT}

for name in defaults:
    text = prompt_texts[name]
    assert isinstance(text, str), f"prompt_texts['{name}']에 프롬프트 문자열을 담으세요."
    assert text == korean_rule + defaults[name], f"{name}: 기본 프롬프트를 고치지 말고 맨 앞에 korean_rule만 붙이세요."

assert set(prompt_paths) == {"extract_graph", "summarize_descriptions", "community_report"}, "prompt_paths에 프롬프트 세 개의 파일 경로를 담으세요."

for name in prompt_paths:
    path = prompt_paths[name]
    assert Path(path).name == f"gwaje01_{name}_ko.txt" and Path(path).exists(), f"{name}: output_dir / f'gwaje01_{{name}}_ko.txt'에 저장하세요."
    assert Path(path).read_text(encoding="utf-8") == prompt_texts[name], f"{name}: 파일 내용이 prompt_texts와 다릅니다. UTF-8로 저장하세요."

print("한국어 프롬프트 확인 완료 | 추출 프롬프트에서 바꾼 문장:", instruction)

## 3. 설정 dict를 완성하고 설정 객체를 만드세요

교안 01에서는 `make_settings`가 설정 dict를 통째로 만들었습니다. 여기서는 모델과 저장 경로만 담긴 `settings`를 제공하고, 나머지 키를 직접 채웁니다. 교안과 같이 원문을 넘기는 검색은 모두 문서 약 5편(`TOP_K`) 분량을 넘기도록 Basic은 청크 수 `k`, Local·DRIFT는 원문 비율 `text_unit_prop`으로 맞춥니다. `text_unit_prop`은 프로젝트 폴더 셀에서 이미 계산해 두었습니다.

| 키 | 넣을 값 |
|---|---|
| `extract_graph` | `entity_types`: 앞에서 정한 목록, `prompt`: 추출 프롬프트 경로(문자열), `max_gleanings`: 0(빠진 개체를 다시 묻는 추가 추출 끄기) |
| `summarize_descriptions` | `prompt`: 설명 요약 프롬프트 경로(문자열) |
| `community_reports` | `graph_prompt`: 커뮤니티 보고서 프롬프트 경로(문자열) |
| `cluster_graph` | `use_lcc`: `False`(가장 큰 연결 묶음만 쓰지 않고 떨어진 묶음도 커뮤니티로 나누기) |
| `basic_search` | `k`: `TOP_K` |
| `local_search` | `text_unit_prop`: `text_unit_prop` |
| `drift_search` | `primer_folds` 1, `drift_k_followups` 3, `n_depth` 1, `local_search_text_unit_prop`: `text_unit_prop`, `local_search_temperature` 1, `reduce_temperature` 1(gpt-5 계열은 1만 받음) |

경로(문자열)는 `prompt_paths`의 `Path`를 `str()`로 바꾼 값입니다.

**작성:** `settings`에 더할 키 일곱 개와 설정 객체 `config`

**목적:** 교안 01의 설정 dict와 같은 구성을 의약품 자료의 개체 유형·프롬프트로 채워 설정 객체를 만듭니다.

**구현할 처리**
1. 표의 키 일곱 개를 `settings`에 더합니다. 값은 키마다 dict입니다(예: `settings["basic_search"] = {"k": TOP_K}`).
2. `GraphRagConfig(**settings)`로 설정 객체를 만들어 `config`에 담습니다. `GraphRagConfig`는 모르는 키를 오류 없이 무시해 기본값으로 색인하므로, 키 이름은 표와 똑같이 적습니다.

**반환값:** `config`(`GraphRagConfig` 객체)

In [ ]:
# 키가 담기므로 settings와 config는 출력하지 않습니다.
api_key = os.environ["OPENAI_API_KEY"]
# 속도 제한에 걸리면 기다렸다가 다시 부릅니다.
retry = {"type": "exponential_backoff"}

# 모델과 저장 경로만 담은 기본 설정입니다. 경로는 모두 문자열로 넣습니다.
settings = {
    # 추출·요약·답변에 쓰는 채팅 모델
    "completion_models": {
        "default_completion_model": {"model_provider": "openai", "model": model_name,
                                     "api_key": api_key, "retry": retry},
    },
    # 임베딩 모델. 벡터를 768차원으로 줄입니다.
    "embedding_models": {
        "default_embedding_model": {"model_provider": "openai", "model": "text-embedding-3-large",
                                    "api_key": api_key, "retry": retry, "call_args": {"dimensions": 768}},
    },
    # 입력·출력·갱신·캐시·로그를 모두 프로젝트 폴더 아래에 둡니다(기본값은 노트북 폴더 기준).
    "input_storage": {"base_dir": str(project_dir / "input")},
    "output_storage": {"base_dir": str(project_dir / "output")},
    # 갱신 색인 때마다 이전 parquet 파일(previous)과 새 문서의 parquet 파일(delta)을 남기는 폴더
    "update_output_storage": {"base_dir": str(project_dir / "update_output")},
    # LLM 응답 캐시: 같은 요청을 다시 보내면 저장된 응답을 씁니다.
    "cache": {"storage": {"base_dir": str(project_dir / "cache")}},
    # 색인 단계별 실행 로그
    "reporting": {"base_dir": str(project_dir / "logs")},
    # 청크·개체 설명·커뮤니티 보고서 임베딩을 저장하는 벡터 저장소(차원은 임베딩과 같은 768)
    "vector_store": {"db_uri": str(project_dir / "output" / "lancedb"), "vector_size": 768},
}

In [ ]:
# [작성] 추출·요약·커뮤니티 보고서·커뮤니티 나누기·검색 설정을 더하고 설정 객체를 만드세요.
from graphrag.config.models.graph_rag_config import GraphRagConfig

# 1. extract_graph: 개체 유형, 추출 프롬프트 경로(문자열), 추가 추출 0회
settings["extract_graph"] = ...

# 2. summarize_descriptions·community_reports: 한국어 프롬프트 경로(문자열)
settings["summarize_descriptions"] = ...
settings["community_reports"] = ...

# 3. cluster_graph: 떨어진 개체 묶음도 커뮤니티로 나누기
settings["cluster_graph"] = ...

# 4. basic_search: 질문과 가까운 청크 TOP_K개
settings["basic_search"] = ...

# 5. local_search·drift_search: 같은 원문 비율, DRIFT 탐색 값과 temperature
settings["local_search"] = ...
settings["drift_search"] = ...

# 6. settings로 설정 객체를 만들어 config에 담으세요.
config = ...

In [ ]:
# [자가채점] 설정 객체에 실제로 들어간 값과 키 이름을 확인합니다. 모델은 호출하지 않습니다.
assert isinstance(config, GraphRagConfig), "GraphRagConfig(**settings)의 결과를 config에 담으세요. `...`를 채웠는지도 확인하세요."

# GraphRagConfig는 모르는 키를 무시하므로 키 이름을 설정 모델의 필드와 직접 대조합니다.
for key in ["extract_graph", "summarize_descriptions", "community_reports", "cluster_graph",
            "basic_search", "local_search", "drift_search"]:
    assert isinstance(settings.get(key), dict), f"settings['{key}']에 dict를 담으세요."
    field_names = type(getattr(config, key)).model_fields
    for name in settings[key]:
        assert name in field_names, f"settings['{key}']의 '{name}'은 없는 설정 이름입니다. 오타를 확인하세요."

extract = config.extract_graph
assert extract.entity_types == entity_types, "extract_graph의 entity_types에 앞에서 정한 목록을 넣으세요."
assert str(extract.prompt) == str(prompt_paths["extract_graph"]), "extract_graph의 prompt에 추출 프롬프트 경로를 넣으세요."
assert extract.max_gleanings == 0, "추가 추출(max_gleanings)은 0으로 끄세요."
assert str(config.summarize_descriptions.prompt) == str(prompt_paths["summarize_descriptions"]), "summarize_descriptions의 prompt에 설명 요약 프롬프트 경로를 넣으세요."
assert str(config.community_reports.graph_prompt) == str(prompt_paths["community_report"]), "community_reports의 graph_prompt에 커뮤니티 보고서 프롬프트 경로를 넣으세요."
assert config.cluster_graph.use_lcc is False, "use_lcc를 False로 두어 떨어진 개체 묶음도 커뮤니티로 나누세요."
assert config.basic_search.k == TOP_K, "Basic은 청크 TOP_K개를 넘기게 하세요."
assert config.local_search.text_unit_prop == text_unit_prop, "Local의 원문 비율에 text_unit_prop을 넣으세요."
drift = config.drift_search
assert drift.local_search_text_unit_prop == text_unit_prop, "DRIFT도 Local과 같은 원문 비율을 쓰게 하세요."
assert (drift.primer_folds, drift.drift_k_followups, drift.n_depth) == (1, 3, 1), "DRIFT 탐색 값은 primer_folds 1, drift_k_followups 3, n_depth 1로 넣으세요."
assert drift.local_search_temperature == 1 and drift.reduce_temperature == 1, "DRIFT의 두 temperature를 1로 두세요."

# 저장 경로가 모두 프로젝트 폴더 아래에 있는지 봅니다.
for path in [config.input_storage.base_dir, config.output_storage.base_dir, config.cache.storage.base_dir]:
    assert Path(path).is_relative_to(project_dir), f"{path}: 저장 경로가 프로젝트 폴더 밖입니다. 제공한 settings 셀부터 다시 실행하세요."

print("설정 확인 완료 | 개체 유형:", extract.entity_types, "| Basic 청크 수:", config.basic_search.k)

# Local이 원문 청크에 쓰는 토큰 = 원문 비율 x 컨텍스트 한도(기본값 12,000)
text_tokens = round(config.local_search.text_unit_prop * config.local_search.max_context_tokens)

print("원문 비율:", config.local_search.text_unit_prop, "| 원문에 쓰는 토큰:", text_tokens, "/", config.local_search.max_context_tokens)

## 4. 검색이 넘긴 원문을 문서 ID로 바꾸세요

교안 01에서 본 대로 각 검색은 답변 모델에 넘긴 컨텍스트를 `{테이블 이름: DataFrame}` dict로 돌려주고, 원문 테이블 `sources`는 청크를 번호로만 가리킵니다. 그래서 `text_units`와 `documents` 테이블을 거쳐야 정답 근거 문서와 비교할 수 있는 문서 ID가 나옵니다.

| 테이블 | 쓰는 열 |
|---|---|
| 컨텍스트의 `sources` | `id`: 청크 번호(문자열). `text_units`의 `human_readable_id`(정수)와 같은 값 |
| `text_units` | `human_readable_id`, `document_id`(문서 내부 id) |
| `documents` | `id`(문서 내부 id), `title`(입력 파일 이름, 예: `drug_195700020.txt`) |

문서 내부 id는 GraphRAG가 문서마다 붙인 긴 문자열이고, 문서 ID는 파일 이름에서 `.txt`를 뗀 `drug_195700020` 같은 값입니다.

**작성:** `context_doc_ids(contexts, tables)` 함수

**목적:** 검색 한 번이 넘긴 컨텍스트 목록을 문서 ID 목록으로 바꿉니다. DRIFT는 후속 Local 검색마다 컨텍스트가 생겨 목록을 받습니다.

**구현할 처리**
1. `text_units`는 `human_readable_id`, `documents`는 `id`를 인덱스로 둡니다(`set_index`).
2. `sources` 테이블이 없는 컨텍스트(Global의 커뮤니티 보고서)는 건너뜁니다.
3. `sources`의 청크 번호를 `int()`로 바꿔 문서 내부 id를 찾고, 그 문서 `title`에서 `.txt`를 뗀 문서 ID를 모읍니다.
4. 중복을 빼고 정렬합니다.

**반환값:** 문서 ID 리스트. 예: `["drug_195700020", "drug_198900672"]`. 컨텍스트가 없거나 모두 `sources`가 없으면 `[]`입니다.

In [ ]:
# [작성] 컨텍스트 목록의 원문 청크를 문서 ID로 바꾸세요.
def context_doc_ids(contexts, tables):
    """run_search가 모은 컨텍스트의 원문 테이블(sources)에 든 청크가 나온 문서 ID를 중복 없이 정렬한 list로 반환합니다.

    contexts: run_search가 반환한 컨텍스트 목록. 원소 하나가 {테이블 이름: DataFrame} dict이며, sources가 없는 컨텍스트(Global의 커뮤니티 보고서)는 건너뜁니다.
    tables: read_tables가 반환한 테이블 dict. text_units·documents 테이블로 청크 번호를 문서 ID로 바꿉니다.
    """
    # 1. text_units·documents 테이블의 인덱스를 정하세요.
    ...

    # 2~3. sources가 있는 컨텍스트에서 청크 번호로 문서 ID를 찾아 모으세요.
    ...

    # 4. 중복 없이 정렬한 리스트를 반환하세요.
    ...

In [ ]:
# [자가채점] 작은 고정 테이블로 청크 번호를 문서 ID로 바꾸는지 확인합니다. 모델은 호출하지 않습니다.
fake_tables = {
    "text_units": pd.DataFrame({"human_readable_id": [0, 1, 2, 3, 4],
                                "document_id": ["d-a", "d-a", "d-b", "d-c", "d-d"]}),
    "documents": pd.DataFrame({"id": ["d-a", "d-b", "d-c", "d-d"],
                               "title": ["drug_a.txt", "drug_b.txt", "drug_c.txt", "drug_d.txt"]}),
}

# 원문이 든 컨텍스트, 커뮤니티 보고서만 든 컨텍스트(Global), 앞과 겹치는 청크와 새 문서 청크가 함께 든 컨텍스트(DRIFT 후속 검색)
fake_contexts = [
    {"sources": pd.DataFrame({"id": ["2", "0"]}), "entities": pd.DataFrame({"id": ["3"]})},
    {"reports": pd.DataFrame({"id": ["3"]})},
    {"sources": pd.DataFrame({"id": ["1", "4"]})},
]

try:
    found = context_doc_ids(fake_contexts, fake_tables)
except KeyError as error:
    raise AssertionError(f"KeyError {error}: sources의 청크 번호는 문자열이라 int()로 바꿔 찾고, sources가 없는 컨텍스트는 건너뛰세요.") from error

assert isinstance(found, list), "문서 ID 리스트를 반환하세요. `...`를 채웠는지도 확인하세요."
assert found == ["drug_a", "drug_b", "drug_d"], "컨텍스트 목록 전체의 sources를 모두 모으세요(DRIFT는 후속 검색마다 컨텍스트가 생깁니다). 청크 번호는 정수로 바꿔 text_units, documents 테이블 순서로 찾고 .txt를 뗀 문서 ID를 중복 없이 정렬합니다."
assert context_doc_ids([{"reports": pd.DataFrame({"id": ["3"]})}], fake_tables) == [], "sources가 없는 컨텍스트는 건너뛰세요."
assert context_doc_ids([], fake_tables) == [], "컨텍스트가 없으면 빈 리스트를 반환하세요."

print("문서 ID 변환 확인 완료")

## 5. 질문 유형마다 검색 방식을 고르세요

**작성:** 검색 계획 dict `SEARCH_PLAN`

**목적:** 교안 01의 질문 유형과 네 가지 검색에서 본 "잘 맞는 질문"을 기준으로, 질문 유형마다 실행할 검색을 정합니다. 방식마다 질문당 한 번씩 검색하므로 이 계획이 비용도 정합니다.

**구현할 처리**
1. 키는 질문 유형 `"single"`(단일 사실), `"multihop"`(멀티홉), `"global"`(전역)이고, 값은 방식 이름(`"basic"`, `"local"`, `"drift"`, `"global"`) 리스트입니다.
2. 단일 사실과 멀티홉에는 비교 기준 `"basic"`과 그래프 검색(`"local"`, `"drift"`) 하나 이상을 함께 넣습니다. 멀티홉에는 후속 질문으로 문서를 이어 읽는 `"drift"`를 넣습니다.
3. 전역은 `"global"`만 씁니다. Global은 원문 없이 커뮤니티 보고서를 모두 읽어 비싸므로 다른 유형에는 넣지 않습니다.

**반환값:** `SEARCH_PLAN`(질문 유형을 키, 방식 이름 리스트를 값으로 하는 dict)

In [ ]:
# [작성] 질문 유형마다 실행할 검색 방식을 고르세요.
SEARCH_PLAN = ...

In [ ]:
# [자가채점] 계획이 조건을 지키는지 확인하고 실행할 검색 횟수를 셉니다.
assert isinstance(SEARCH_PLAN, dict), "SEARCH_PLAN에 dict를 담으세요. `...`를 채웠는지도 확인하세요."
assert set(SEARCH_PLAN) == {"single", "multihop", "global"}, "키는 single, multihop, global 세 개입니다."

for qtype in SEARCH_PLAN:
    modes = SEARCH_PLAN[qtype]
    assert isinstance(modes, list) and modes, f"{qtype}: 방식 이름 리스트를 담으세요."
    assert len(set(modes)) == len(modes), f"{qtype}: 같은 방식을 두 번 넣지 마세요."
    assert set(modes) <= {"basic", "local", "drift", "global"}, f"{qtype}: 방식은 basic, local, drift, global 가운데에서 고르세요."

assert SEARCH_PLAN["global"] == ["global"], "전역 질문은 global만 씁니다."

for qtype in ["single", "multihop"]:
    modes = SEARCH_PLAN[qtype]
    assert "global" not in modes, f"{qtype}: Global은 원문을 넘기지 않고 비쌉니다. 이 유형에는 넣지 마세요."
    assert "basic" in modes and len(modes) >= 2, f"{qtype}: 기준선 basic과 그래프 검색 하나 이상을 함께 실행하세요."

assert "drift" in SEARCH_PLAN["multihop"], "멀티홉에는 문서를 이어 읽는 drift를 넣으세요."
planned_runs = 0

for question in questions:
    planned_runs += len(SEARCH_PLAN[question["qtype"]])

print("검색 계획 확인 완료 | 실행할 검색:", planned_runs, "번")

## 6. 색인을 실행하고 사용량을 기록하세요

`api.build_index`는 청크 나누기, 개체·관계 추출, 커뮤니티 나누기, 커뮤니티 보고서 쓰기, 임베딩을 차례로 실행합니다. 여기서부터 모델을 실제로 호출하며 몇 분 걸립니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `usage`, `record_chat_call` | litellm 성공 콜백 `record_chat_call`이 채팅 호출이 성공할 때마다 호출 수와 입력·출력 토큰을 `usage` dict에 더합니다. |
| `read_tables(project_dir)` | 색인 결과 parquet 파일 여섯 개를 `{테이블 이름: DataFrame}` dict로 읽어 반환합니다. |

**작성:** 색인 결과 `index_results`와 사용량 기록 `index_usage`

**목적:** 교안 01의 `build_index` 실행과 비용 기록처럼 색인을 만들고, 갱신과 비교할 사용량을 남깁니다.

**구현할 처리**
1. `usage`의 세 값을 0으로 되돌립니다.
2. `await api.build_index(config=config, method="standard")`의 결과를 `index_results`에 담습니다.
3. 단계마다 `error`가 `None`이면 완료와 `workflow`를, 아니면 실패와 `workflow`·`error`를 출력합니다.
4. `usage`에 `seconds` 키(`round(index_seconds, 1)`)를 더한 새 dict를 `index_usage`에 담습니다.

**반환값:** `index_results`(단계별 결과 리스트), `index_usage`(`llm_calls` 호출 수, `input_tokens` 입력 토큰, `output_tokens` 출력 토큰, `seconds` 소수 첫째 자리까지 반올림한 초의 네 키를 가진 dict)

In [ ]:
import litellm

# 채팅 호출 수와 토큰을 더해 갈 dict
usage = {"llm_calls": 0, "input_tokens": 0, "output_tokens": 0}


async def record_chat_call(kwargs, response, start_time, end_time):
    """채팅 호출 하나가 성공하면 usage의 llm_calls를 1 늘리고 input_tokens·output_tokens에 이번 토큰 수를 더합니다.

    litellm.success_callback에 등록하면 litellm이 호출이 성공할 때마다 부릅니다. 임베딩 호출은 LLM 호출이 아니라 세지 않습니다.
    네 인자는 litellm이 넘기는 값이며, kwargs(호출 인자, call_type으로 호출 종류 확인)와 response(응답, response.usage에 토큰 수)만 씁니다.
    """
    # kwargs["call_type"]이 호출 종류입니다. GraphRAG는 채팅을 비동기 acompletion으로 부르며,
    # 임베딩 호출(aembedding)은 LLM 호출이 아니라 세지 않습니다.
    if kwargs["call_type"] == "acompletion":
        usage["llm_calls"] += 1
        # response.usage에 이번 호출의 입력(prompt)·출력(completion) 토큰 수가 있습니다.
        usage["input_tokens"] += response.usage.prompt_tokens
        usage["output_tokens"] += response.usage.completion_tokens


# 호출이 성공할 때마다 litellm이 이 함수를 부르도록 등록합니다.
litellm.success_callback = [record_chat_call]


def read_tables(project_dir):
    """project_dir/output에 저장된 GraphRAG 색인 결과 parquet 파일 여섯 개를 읽어 {테이블 이름: DataFrame} dict로 반환합니다.

    테이블 이름: documents, text_units, entities, relationships, communities, community_reports
    """
    tables = {}

    # 색인 결과는 project_dir/output에 '테이블 이름.parquet' 파일로 저장됩니다.
    for name in ["documents", "text_units", "entities", "relationships", "communities", "community_reports"]:
        tables[name] = pd.read_parquet(project_dir / "output" / f"{name}.parquet")

    return tables

In [ ]:
# [작성] 색인을 실행하고 사용량을 기록하세요.
import asyncio
import time

import graphrag.api as api

# 1. usage의 세 값을 0으로 되돌리세요.
...

# 2. config로 api.build_index를 method="standard"로 실행해 index_results에 담으세요. 비동기 함수라 await를 붙입니다.
start = time.time()
index_results = ...
index_seconds = time.time() - start
# litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
await asyncio.sleep(1)

# 3. 단계(workflow)마다 성공과 실패(error)를 출력하세요.
...

# 4. usage에 걸린 초를 seconds 키로 더한 새 dict를 index_usage에 담으세요.
index_usage = ...

In [ ]:
# [자가채점] 색인이 만든 테이블과 사용량 기록을 확인합니다. 모델은 호출하지 않습니다.
assert isinstance(index_results, list) and index_results, "await api.build_index(...)의 결과를 index_results에 담으세요."

for result in index_results:
    assert result.error is None, f"{result.workflow} 단계가 실패했습니다: {result.error!r}. 프로젝트 폴더를 만드는 셀부터 다시 실행하세요."

assert isinstance(index_usage, dict) and "seconds" in index_usage, "usage에 seconds를 더한 새 dict를 index_usage에 담으세요."
assert index_usage["llm_calls"] > 0, "LLM 호출이 0회로 기록됐습니다. 사용량 콜백 셀을 실행했는지, usage를 색인 전에 0으로 되돌렸는지 확인하세요. 다시 색인할 때는 캐시를 지우도록 프로젝트 폴더를 만드는 셀부터 실행합니다."

# 색인이 만든 parquet 파일을 테이블로 읽어 행 수를 봅니다.
tables = read_tables(project_dir)
assert len(tables["documents"]) == len(docs_table), "documents 테이블의 행 수가 입력 문서 수와 다릅니다. 프로젝트 폴더를 만드는 셀부터 다시 실행하세요."
assert len(tables["entities"]) > 0 and len(tables["community_reports"]) > 0, "개체나 커뮤니티 보고서가 없습니다. 추출 프롬프트와 entity_types를 확인하세요."

print("색인 확인 완료 | 시간:", index_usage["seconds"], "초 | LLM 호출:", index_usage["llm_calls"], "회 | 입력 토큰:", index_usage["input_tokens"], "| 출력 토큰:", index_usage["output_tokens"])

for name in tables:
    print(name, len(tables[name]), "행")

In [ ]:
# LLM이 개체마다 붙인 유형(type)별 개수
display(tables["entities"]["type"].value_counts())

멀티홉 질문의 다리(제조사·성분)가 개체로 뽑혔는지 봅니다. `degree`는 그 개체에 이어진 관계 수로, 여러 제품과 이어져 있어야 그래프로 건너갈 수 있습니다.

In [ ]:
entities = tables["entities"]
# 자료 설명의 공유 제조사·성분 이름이 든 개체를 고릅니다(표기가 조금 달라도 고르도록 포함 여부로 봅니다).
bridge_names = ["동화약품", "현대약품", "아세트아미노펜", "카르보시스테인"]
is_bridge = entities["title"].str.contains("|".join(bridge_names))

display(entities[is_bridge][["title", "type", "degree", "description"]])

## 7. 골드 질문 8개를 검색하고 결과 표를 저장하세요

모델을 호출하며 몇 분 걸립니다. 아래 제공 셀의 두 함수는 교안 01과 같은 코드입니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `run_search(mode, question, config, tables)` | `mode`(`basic`·`local`·`global`·`drift`) 검색을 실행해 `(답, 콜백이 모은 컨텍스트 목록, 걸린 초)`를 반환합니다. 시작할 때 `usage`를 0으로 되돌리므로, 끝난 뒤 `usage`에는 이 검색의 사용량만 남습니다. |
| `make_result_row(method, question, given_ids, usage, seconds)` | 결과 표 한 행(dict)을 만듭니다. 원문을 넘기지 않는 Global은 `given_ids`에 `None`을 넘깁니다. |

**작성:** 결과 행 목록 `result_rows`, 답 dict `answers`, 결과 표 `results_table`

**목적:** 교안 01에서 질문 하나씩 실행해 본 네 검색을 골드 질문 8개 전체로 넓혀, 넘긴 문서와 비용을 한 표로 비교합니다.

**구현할 처리**
1. 질문마다 `SEARCH_PLAN[question["qtype"]]`의 방식으로 `run_search`를 한 번씩 실행합니다.
2. `given_ids`는 `global`이면 `None`, 아니면 `context_doc_ids(contexts, tables)`입니다.
3. `make_result_row("graphrag_" + mode, question, given_ids, usage, seconds)`를 `result_rows`에 더하고, 답을 `answers[(question["id"], mode)]`에 담습니다.
4. `result_rows`로 만든 DataFrame을 `results_table`에 담고 `output/gwaje01_results.csv`로 저장합니다(`index=False`).

**반환값:** `results_table`(검색 한 번이 한 행인 DataFrame), `answers`(키는 `(질문 ID, 방식)` 튜플, 값은 답 문자열인 dict). 예: `answers[("S1", "basic")]`가 S1을 Basic으로 검색한 답입니다.

DRIFT의 첫 답이 정해진 JSON 형식에 맞지 않아 가끔 에러가 납니다. 그때는 이 셀을 다시 실행하세요(끝난 검색도 다시 실행됩니다).

In [ ]:
from graphrag.callbacks.query_callbacks import QueryCallbacks

# 레벨 0·1 커뮤니티 보고서를 씁니다.
community_level = 1
# 답 형식 지시: 전역 질문은 항목을 모두 나열하게, 나머지는 짧게 답하게 합니다.
SHORT_ANSWER = "한국어로 한두 문장. 근거에 없으면 모른다고 답한다"
LIST_ANSWER = "한국어로, 해당하는 항목을 모두 나열한 목록"


class ContextCollector(QueryCallbacks):
    """GraphRAG 검색에 넘기는 콜백입니다. 검색이 답변 모델에 넘길 컨텍스트가 만들어질 때마다 contexts 목록에 쌓습니다.

    contexts의 원소 하나는 {테이블 이름: DataFrame} dict입니다. 검색이 끝난 뒤 contexts를 읽으면 그 검색이 넘긴 컨텍스트를 모두 볼 수 있습니다.
    """

    def __init__(self):
        # 컨텍스트(테이블 이름 -> DataFrame dict)를 불린 순서대로 쌓습니다.
        self.contexts = []

    def on_context(self, context):
        """만들어진 컨텍스트({테이블 이름: DataFrame} dict)를 contexts 끝에 덧붙입니다."""
        # 컨텍스트가 만들어질 때마다 불립니다. DRIFT는 후속 Local 검색마다 불립니다.
        self.contexts.append(context)


async def run_search(mode, question, config, tables):
    """질문 하나에 GraphRAG 검색 하나(mode)를 실행해 (answer, contexts, seconds)를 반환합니다.

    mode: "basic", "local", "global", "drift" 가운데 하나
    question: 골드 질문 dict. qtype이 "global"이면 항목을 모두 나열하게, 나머지는 한두 문장으로 답하게 합니다.
    config: GraphRAG 설정 객체(GraphRagConfig)
    tables: read_tables가 반환한 테이블 dict
    반환값: answer는 답변 글, contexts는 검색이 답변 모델에 넘긴 컨텍스트 목록(ContextCollector가 모은 것), seconds는 검색에 걸린 시간(초)
    시작할 때 usage를 0으로 되돌리므로, 끝난 뒤 usage에는 이 검색 한 번의 사용량만 남습니다.
    """
    # 검색마다 새 콜백을 만들어 앞 검색의 컨텍스트가 섞이지 않게 합니다.
    collector = ContextCollector()

    # 전역 질문은 항목을 모두 나열하게, 나머지는 짧게 답하게 합니다.
    if question["qtype"] == "global":
        response_type = LIST_ANSWER
    else:
        response_type = SHORT_ANSWER

    # 이 검색의 사용량만 세도록 0으로 되돌립니다.
    usage.update(llm_calls=0, input_tokens=0, output_tokens=0)
    start = time.time()

    # 반환값은 (답, 마지막 컨텍스트)입니다. 컨텍스트는 콜백이 모은 목록을 쓰므로 버립니다.
    if mode == "basic":
        # Basic: text_units 테이블만 넘기고, 질문과 가까운 청크를 벡터 저장소에서 찾습니다.
        answer, _ = await api.basic_search(
            config=config, text_units=tables["text_units"],
            response_type=response_type, query=question["question"], callbacks=[collector])
    elif mode == "local":
        # Local: 개체·관계·커뮤니티·커뮤니티 보고서·청크 테이블을 모두 씁니다. 주장(covariates) 테이블은 만들지 않아 None입니다.
        answer, _ = await api.local_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], text_units=tables["text_units"],
            relationships=tables["relationships"], covariates=None, community_level=community_level,
            response_type=response_type, query=question["question"], callbacks=[collector])
    elif mode == "global":
        # Global: 커뮤니티·커뮤니티 보고서 테이블로 map-reduce를 합니다. 원문 청크 테이블은 넘기지 않습니다.
        answer, _ = await api.global_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], community_level=community_level,
            # False: 레벨 상한 이하 커뮤니티 보고서를 모두 읽습니다. True면 LLM이 관련 있는 것만 골라 읽습니다.
            dynamic_community_selection=False, response_type=response_type, query=question["question"],
            callbacks=[collector])
    elif mode == "drift":
        # DRIFT: 커뮤니티 보고서로 첫 답을 만들고 후속 질문마다 Local 검색을 해서 Local과 같은 테이블이 필요합니다.
        answer, _ = await api.drift_search(
            config=config, entities=tables["entities"], communities=tables["communities"],
            community_reports=tables["community_reports"], text_units=tables["text_units"],
            relationships=tables["relationships"], community_level=community_level,
            response_type=response_type, query=question["question"], callbacks=[collector])

    seconds = time.time() - start
    # litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
    await asyncio.sleep(1)
    return answer, collector.contexts, seconds

In [ ]:
def evidence_hit(given_ids, gold_ids):
    """정답 근거 문서 가운데 검색이 넘긴 문서가 몇 편인지 세어 int로 반환합니다.

    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록
    gold_ids: 질문의 정답 근거 문서 ID 목록(골드 질문의 gold_doc_ids)
    예: 정답 근거 3편 가운데 2편을 넘겼으면 2를 반환합니다.
    """
    hit = 0

    # 정답 근거 문서마다 넘긴 목록에 들어 있는지 봅니다.
    for gold_id in gold_ids:
        if gold_id in given_ids:
            hit += 1

    return hit


def make_result_row(method, question, given_ids, usage, seconds):
    """검색 한 번의 결과를 비교용 결과 표의 한 행(dict)으로 만들어 반환합니다.

    method: 검색 방식 이름(예: "graphrag_local", "lightrag_mix")
    question: 골드 질문 dict(id, qtype, gold_doc_ids)
    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록. 원문을 넘기지 않는 GraphRAG Global은 None을 넘기고, 이때 문서 열은 빈칸(None)이 됩니다.
    usage: 이 검색의 LLM 사용량 dict(llm_calls 호출 수, input_tokens 입력 토큰, output_tokens 출력 토큰)
    seconds: 검색에 걸린 시간(초)
    문서 열: docs_given(넘긴 문서 수), gold_docs(정답 근거 수), hit_docs(정답 근거 가운데 넘긴 수), all_hit(모두 넘겼으면 1, 아니면 0)
    """
    # 사용량과 시간은 모든 방식에 채우고, 문서 열은 빈칸(None)으로 시작합니다.
    row = {"method": method, "question_id": question["id"], "qtype": question["qtype"],
           "docs_given": None, "gold_docs": None, "hit_docs": None, "all_hit": None,
           "llm_calls": usage["llm_calls"], "input_tokens": usage["input_tokens"],
           "output_tokens": usage["output_tokens"], "seconds": round(seconds, 1)}

    # 원문 문서를 넘긴 방식만 문서 열을 채웁니다.
    if given_ids is not None:
        gold_ids = question["gold_doc_ids"]
        hit = evidence_hit(given_ids, gold_ids)
        row["docs_given"] = len(given_ids)
        row["gold_docs"] = len(gold_ids)
        row["hit_docs"] = hit

        # 정답 근거를 모두 넘겼으면 1
        if hit == len(gold_ids):
            row["all_hit"] = 1
        else:
            row["all_hit"] = 0

    return row

In [ ]:
# [작성] 질문마다 계획한 방식으로 검색하고 결과 행과 답을 모아 저장하세요.
result_rows = []
answers = {}

for question in questions:
    for mode in SEARCH_PLAN[question["qtype"]]:
        # 1. run_search로 검색하세요. 반환값은 (답, 컨텍스트 목록, 걸린 초)입니다.
        ...

        # 2. global이면 given_ids를 None으로, 아니면 context_doc_ids로 넘긴 문서 ID를 구하세요.
        ...

        # 3. make_result_row로 만든 행을 result_rows에 더하고, 답을 answers[(질문 ID, mode)]에 담으세요.
        ...

        print(question["id"], mode, "| LLM 호출:", usage["llm_calls"], "회")

# 4. result_rows로 만든 DataFrame을 results_table에 담고 output/gwaje01_results.csv로 저장하세요.
results_table = ...
...

In [ ]:
# [자가채점] 결과 표의 행 수·열과 저장한 파일을 확인합니다. 모델은 호출하지 않습니다.
assert isinstance(results_table, pd.DataFrame), "result_rows로 만든 DataFrame을 results_table에 담으세요. `...`를 채웠는지도 확인하세요."
expected_columns = ["method", "question_id", "qtype", "docs_given", "gold_docs", "hit_docs", "all_hit",
                    "llm_calls", "input_tokens", "output_tokens", "seconds"]
assert list(results_table.columns) == expected_columns, "make_result_row가 만든 행을 그대로 쌓으세요."
assert len(results_table) == planned_runs, f"질문마다 계획한 방식을 모두 실행하면 {planned_runs}행입니다."
assert set(results_table["method"]) <= {"graphrag_basic", "graphrag_local", "graphrag_drift", "graphrag_global"}, "method는 'graphrag_' + mode 입니다."
global_rows = results_table[results_table["qtype"] == "global"]
assert global_rows["docs_given"].isna().all(), "Global은 원문을 넘기지 않으므로 given_ids에 None을 넘기세요."
basic_rows = results_table[results_table["method"] == "graphrag_basic"]
assert (basic_rows["docs_given"] > 0).all(), "Basic이 넘긴 문서가 0편입니다. context_doc_ids에 run_search가 돌려준 컨텍스트 목록을 넘기세요."
graph_rows = results_table[results_table["qtype"] != "global"]
assert graph_rows["docs_given"].notna().all(), "Global이 아닌 검색은 context_doc_ids로 구한 문서 ID를 넘기세요."
assert (results_table["llm_calls"] > 0).all(), "LLM 호출이 0회인 검색이 있습니다. 사용량 콜백 셀을 실행했는지 확인하세요."
assert len(answers) == planned_runs, "검색마다 답을 answers[(질문 ID, mode)]에 담으세요."

for question_id, mode in answers:
    assert ("graphrag_" + mode) in set(results_table[results_table["question_id"] == question_id]["method"]), "answers의 키는 (질문 ID, mode)입니다."

saved = pd.read_csv(output_dir / "gwaje01_results.csv")
assert list(saved.columns) == expected_columns and len(saved) == len(results_table), "results_table을 output/gwaje01_results.csv로 저장하세요(index=False)."

print("결과 표 확인 완료:", len(results_table), "행")

In [ ]:
# 유형·방식별 평균: 정답 근거를 모두 넘긴 비율(all_hit), 넘긴 문서 수, 질문 1개당 호출 수와 입력 토큰
summary = results_table.groupby(["qtype", "method"])[["all_hit", "docs_given", "llm_calls", "input_tokens"]].mean().round(2)

display(summary)

멀티홉은 질문별로 봅니다. 정답 근거 두 편 가운데 넘긴 수(`hit_docs`)를 넘긴 문서 수(`docs_given`)와 함께 읽으세요.

In [ ]:
# 멀티홉 질문의 행만 골라 질문·방식별로 봅니다.
multihop_rows = results_table[results_table["qtype"] == "multihop"]

display(multihop_rows[["question_id", "method", "docs_given", "hit_docs", "all_hit", "llm_calls", "input_tokens"]])

G1의 Global 답변입니다. 정답 근거 문서의 제품이 답 목록에 빠짐없이 들었는지 확인하세요.

In [ ]:
from IPython.display import Markdown

# Global은 원문을 넘기지 않으므로 정답 근거 문서의 제품 이름을 답과 견줍니다.
g1_gold_titles = docs_table[docs_table["doc_id"].isin(questions_by_id["G1"]["gold_doc_ids"])]["title"].tolist()

print("G1 정답 근거 문서의 제품:", g1_gold_titles)
display(Markdown(answers[("G1", "global")]))

## 8. 새 안내문 한 편을 추가해 색인을 갱신하세요

`extra.json`이 가리키는 안내문은 부광약품의 가래약(성분 카르보시스테인)입니다. 이미 색인된 부광약품 소화제, 그리고 같은 성분의 다른 가래약과 이어집니다.

교안 01에서 본 대로 `is_update_run=True`로 `build_index`를 부르면, `input/`에서 이전 색인에 없던 파일 이름만 찾아 그 문서의 개체·관계를 뽑고, 그 개체·관계로 새 커뮤니티와 커뮤니티 보고서를 써서 기존 테이블에 덧붙입니다. 교안에서 다루지 않은 점이 하나 있습니다. 새 개체가 이름이 같은 기존 개체와 합쳐지면, 그 개체의 설명은 LLM이 다시 요약합니다.

**작성:** 갱신 결과 `update_results`와 사용량 기록 `update_usage`

**목적:** 교안 01의 문서 한 편 추가(갱신)를 의약품 자료로 실행해, 처음 색인과 비교할 사용량을 남깁니다.

**구현할 처리**
1. `data/과제_01/docs/{extra_id}.txt`를 `input_dir`로 복사합니다(`shutil.copy`).
2. `usage`의 세 값을 0으로 되돌립니다.
3. `await api.build_index(config=config, method="standard", is_update_run=True)`의 결과를 `update_results`에 담습니다.
4. `usage`에 `seconds` 키(`round(update_seconds, 1)`)를 더한 새 dict를 `update_usage`에 담습니다.

**반환값:** `update_results`(단계별 결과 리스트), `update_usage`(`index_usage`와 같은 네 키의 dict)

In [ ]:
# [작성] 새 문서를 input에 넣고 갱신 색인을 실행하세요.
# 1. extra_id 문서를 input_dir로 복사하세요.
...

# 2. usage의 세 값을 0으로 되돌리세요.
...

# 3. is_update_run=True로 build_index를 실행해 update_results에 담으세요.
start = time.time()
update_results = ...
update_seconds = time.time() - start
# litellm 사용량 콜백은 호출 뒤 따로 돌아서 잠시 기다립니다.
await asyncio.sleep(1)

# 4. usage에 걸린 초를 seconds 키로 더한 새 dict를 update_usage에 담으세요.
update_usage = ...

In [ ]:
# [자가채점] 새 문서가 테이블에 합쳐졌는지와 갱신 사용량을 확인합니다. 모델은 호출하지 않습니다.
assert (input_dir / f"{extra_id}.txt").exists(), "extra_id 문서를 input_dir로 복사하세요."
assert isinstance(update_results, list) and update_results, "await api.build_index(..., is_update_run=True)의 결과를 update_results에 담으세요."

for result in update_results:
    assert result.error is None, f"{result.workflow} 단계가 실패했습니다: {result.error!r}"

assert isinstance(update_usage, dict) and "seconds" in update_usage, "usage에 seconds를 더한 새 dict를 update_usage에 담으세요."
assert update_usage["llm_calls"] > 0, "갱신 LLM 호출이 0회로 기록됐습니다. usage를 갱신 전에 0으로 되돌렸는지 확인하세요."
updated_tables = read_tables(project_dir)
updated_ids = set(updated_tables["documents"]["title"].str.removesuffix(".txt"))
assert extra_id in updated_ids and len(updated_ids) == len(docs_table) + 1, "갱신 뒤 documents 테이블에 새 문서가 들어 있어야 합니다. method와 is_update_run 인자를 확인하세요."

print("갱신 확인 완료 | 문서:", len(updated_ids), "편")

In [ ]:
# 처음 색인과 갱신의 사용량을 한 DataFrame으로 나란히 비교합니다.
display(pd.DataFrame([index_usage, update_usage], index=["index", "update"]))

새 제품이 기존 제조사·성분 개체에 이어졌는지 `relationships` 테이블에서 봅니다.

In [ ]:
relationships = updated_tables["relationships"]
# 새 문서의 제조사·성분이 든 관계를 고릅니다.
mentions = relationships["source"].str.contains("부광|카르보시스테인") | relationships["target"].str.contains("부광|카르보시스테인")

display(relationships[mentions][["source", "target", "description"]])

## 결과를 읽고 정리하세요

**작성:** 서술형 답안. 숫자는 자신이 실행한 결과 표에서 인용하세요.

**목적:** 결과 표의 숫자로 질문 유형마다 쓸 검색과 그 비용을 정리합니다.

**설명할 내용**
1. 단일 사실(S1~S3): 방식마다 정답 근거를 모두 넘겼는지(`all_hit`)와 질문 1개당 호출 수·입력 토큰을 비교하고, 이 유형에 쓸 방식을 고르세요.
2. 멀티홉(M1~M3): 근거 두 편을 모두 넘긴 방식은 무엇입니까? 넘긴 문서 수(`docs_given`)와 위에서 출력한 다리 개체(제조사·성분) 표를 함께 보고, 제조사·성분이 개체로 뽑힌 것이 결과와 어떻게 이어지는지 설명하세요.
3. 전역(G1·G2): Global 답변이 정답 근거 문서의 제품을 빠짐없이 담았는지, 입력 토큰이 다른 방식보다 얼마나 큰지 쓰세요. 18편 전체 토큰(`total_tokens`)과 비교해, 이 규모의 자료라면 GraphRAG 대신 무엇을 먼저 검토하겠습니까? 18편 전체가 답변 모델 한 번에 넣을 수 있는 양인지부터 보세요.
4. 갱신: 문서 한 편 추가에 든 호출 수를 처음 색인과 비교하고, 갱신만 거듭할 때 생기는 문제를 쓰세요.

*(여기에 확인한 근거와 판단 이유를 작성하세요.)*